![Practicum AI Logo image](https://github.com/PracticumAI/practicumai.github.io/blob/main/images/logo/PracticumAI_logo_250x50.png?raw=true) <img src="images/practicumai_transfer_learning.png" alt="Practicum AI: Transfer Learning icon" align="right" width=50>
***

# Data Preprocessing

This notebook is not part of the course content proper, but it contains some details about how we processed the data for the [01_transfer_learning
_fine_tuning.ipynb notebook](01_transfer_learning_fine_tuning.ipynb). You can safely skip this notebook if you are not interested in the data preprocessing steps.

If you *are* interested in the data preprocessing steps, here are some details about how we processed the data for the transfer learning notebook:


In [ ]:
import os # Used for file handling
import pandas as pd # Used for data manipulation and analysis
import random # Used for random sampling of data


## Step 1: Import Libraries and Setup

Import the necessary libraries for data manipulation and file handling.


In [ ]:
data_path = 'data'

train_path = os.path.join(data_path, "agri_net_train")
val_path = os.path.join(data_path, "agri_net_val")
test_path = os.path.join(data_path, "agri_net_test")



## Step 2: Define Data Paths

Set up the paths to the training, validation, and test datasets.


In [ ]:
# Check train, val, and test directories for folders with spaces or parentheses in the their names
# fix these issues by renaming the folders
# Note: This script assumes that the directory structure is consistent and that the directories are not deeply nested.
# It will rename directories in the specified paths if they contain spaces or parentheses.

for path in [train_path, val_path, test_path]:
    for root, dirs, files in os.walk(path):
        for dir_name in dirs:
            if ' ' in dir_name or '(' in dir_name or ')' in dir_name:
                print(f"Directory '{dir_name}' in '{root}' contains spaces or parentheses.")
            if ' ' in dir_name:
                new_dir_name = dir_name.replace(' ', '_')
                os.rename(os.path.join(root, dir_name), os.path.join(root, new_dir_name))
                print(f"Renamed '{dir_name}' to '{new_dir_name}' in '{root}'")
            if '(' in dir_name or ')' in dir_name:
                new_dir_name = dir_name.replace('(', '').replace(')', '')
                os.rename(os.path.join(root, dir_name), os.path.join(root, new_dir_name))
                print(f"Renamed '{dir_name}' to '{new_dir_name}' in '{root}'")  


## Step 3: Clean Directory Names

Remove spaces and parentheses from directory names to ensure compatibility with different systems and avoid potential issues with file paths.


In [ ]:
# Ensure that train, val, and test directories have consistent folder names within each path
# This script assumes that the directory structure is consistent and that the directories are not deeply nested.

# Get list of all directories in the train, val, and test paths
train_dirs = sorted(os.listdir(train_path))
val_dirs = sorted(os.listdir(val_path))
test_dirs = sorted(os.listdir(test_path))
# Check if the directories in train, val, and test paths are consistent

if train_dirs != val_dirs or train_dirs != test_dirs:
    print("Inconsistent directories found:")
    print(f"Train directories: {train_dirs}")
    print(f"Validation directories: {val_dirs}")
    print(f"Test directories: {test_dirs}")
else:
    print("All directories are consistent across train, val, and test paths.")
    print(f"Directories: {train_dirs}")


## Step 4: Verify Directory Consistency

Check that the same crop categories exist across all three data splits (train, validation, and test).


In [ ]:
# Go through the directories and check the 1st word of each directory name.
# If there is only one directory with that name, print it out

def check_unique_directories(path):
    # Get list of all directories in the path
    dirs = sorted(os.listdir(path))
    # Create a dictionary to count occurrences of each directory name
    dir_count = {}
    for dir_name in dirs:
        first_word = dir_name.split('_')[0]
        if first_word not in dir_count:
            dir_count[first_word] = 1
        else:
            dir_count[first_word] += 1

    # Print unique directory names
    for dir_name, count in dir_count.items():
        if count == 1:
            print(f"Unique directory: {dir_name} in {path}")

            # Unless the directory is "Background"            
# Check unique directories in train, val, and test paths
check_unique_directories(train_path)
check_unique_directories(val_path)
check_unique_directories(test_path)

## Step 5: Identify Problematic Directories

Check for directories that appear in only one data split or that may have issues with the naming convention. This helps identify categories that need manual review or removal.


* I manually deleted the directories for Bael, Basil, Blueberry, Raspberry and Squash from train, val, test dirs.
* I manually moved images from `Lemon_Healthy/[subfolder]` and `Lemon_Diseased/[Subfolder]` directories into `Lemon_Healthy` and `Lemon_Diseased`, and renamed those to `Citrus_[Healthy, Diseased]`.
* Renamed `Coffee_Health` to `Coffee_Healthy`
* `Rice_Bacterial_Leaf_Blight` only has 28 images and there are no healthy rice images. Remove all rice folders.
* Soybean also seems not to fit the mold of others, with no disease states. Remove those.

## Step 6: Manual Data Cleanup

Based on the analysis above, several directories were manually cleaned up:


In [ ]:
# For each directory in train, val, and test paths, check how many images are in each directory
# and print the count
# This script assumes that the directory structure is consistent and that the directories are not deeply nested.

# Create a dataframe to store the counts and directory names
image_counts = []
for path in [train_path]:
    for root, dirs, files in os.walk(path):
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            image_count = len([f for f in os.listdir(dir_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
            image_counts.append({'Directory': dir_name, 'Path': path, 'Image Count': image_count})
            if image_count < 100:
                print(f"Warning: Directory '{dir_path}' in '{path}' contains less than 100 images ({image_count} images).")

# Create a DataFrame from the image counts
df = pd.DataFrame(image_counts)
df


## Step 7: Analyze Image Counts Per Category

Count the number of images in each category across the training, validation, and test sets to identify any classes with low sample sizes.


The smallest traning folder is 58 images in `data/agri_net_train/Tomato_Gray_Spot`. Let's subsample so that each folder has at most 100 images.

In [ ]:
# For the training folder, make a subsample of each category, randomly selecting 100 images
# Put the subsampled images into their corresponding folders in data/agri_net_train100 

# Create the new directory for the subsampled images
subsampled_train_path = os.path.join(data_path, "agri_net_train100")
os.makedirs(subsampled_train_path, exist_ok=True)
# Create the subdirectories for each category
for dir_name in train_dirs:
    os.makedirs(os.path.join(subsampled_train_path, dir_name), exist_ok=True)
# Go through the directories and randomly select 100 images from each directory

for dir_name in train_dirs:
    dir_path = os.path.join(train_path, dir_name)
    image_files = [f for f in os.listdir(dir_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(image_files)
    selected_files = image_files[:100]
    for file_name in selected_files:
        src_path = os.path.join(dir_path, file_name)
        dst_path = os.path.join(subsampled_train_path, dir_name, file_name)
        os.link(src_path, dst_path)  # Create a hard link to the original file

# Check the number of images in each directory in the subsampled train path
subsampled_image_counts = []
for root, dirs, files in os.walk(subsampled_train_path):
    for dir_name in dirs:
        dir_path = os.path.join(root, dir_name)
        image_count = len([f for f in os.listdir(dir_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        subsampled_image_counts.append({'Directory': dir_name, 'Path': subsampled_train_path, 'Image Count': image_count})
# Create a DataFrame from the subsampled image counts
subsampled_df = pd.DataFrame(subsampled_image_counts)
subsampled_df

## Step 8: Subsample Training Data

To balance the training dataset, randomly subsample each category to a maximum of 100 images. This creates a more balanced and manageable training set while maintaining consistency across categories. The subsampled data is saved to a new directory (`agri_net_train100`).


In [ ]:
subsampled_df.describe()

## Preprocessing Complete

The data preprocessing steps are now complete. The cleaned and subsampled training data is ready for use in the transfer learning models.


OK, now we have a training set with up to 100 images per class and clean classes with healthy and one or more disease states.